In [ ]:
import plotly.express as px
from google.cloud import bigquery
import pandas as pd
import seaborn as sns
from utils import read_config

config = read_config.load_config()

In [ ]:
namespaces = config['bigquery']['namespaces']
OUTPUT_NAMESPACE = namespaces['output']
OMOP_NAMESPACE = namespaces['omop']
UNIONED_EHR_NAMESPACE = namespaces['unioned_ehr']
MAPPING_NAMESPACE = namespaces['mappings']
SURVEY_NAMESPACE = namespaces['survey']
ID_CROSSWALK_TABLE = config['bigquery']['tables']['id_crosswalk']
COHORT_TABLE = config['analysis_configs']['COHORT_TABLE']
COHORT_LABEL_PREFIX = config['analysis_configs']['COHORT_LABEL_PREFIX']
ANALYSIS_END_DATE = config['omop_table_configs']['ANALYSIS_END_DATE']
OMOP_TABLE_PREFIX = config['omop_table_configs']['OMOP_TABLE_PREFIX']

In [ ]:
client = bigquery.Client()

#### Loyalty Score with Charlson Commorbidity Index

In [ ]:
query = f"""WITH cci AS
(
SELECT DISTINCT sm.hpo_id AS src_hpo_id,
                cs.diag_category_id,
                cs.diag_category_name,
                count(distinct cohort.person_id) as pt_cnt
FROM `{OMOP_NAMESPACE}.person` AS cohort
INNER JOIN `{OMOP_NAMESPACE}.condition_occurrence` AS condition
    ON cohort.person_id = condition.person_id
INNER JOIN `{OMOP_NAMESPACE}.condition_occurrence_ext` AS c_ext
    ON condition.condition_occurrence_id = c_ext.condition_occurrence_id
INNER JOIN `{MAPPING_NAMESPACE}.site_maskings` sm
ON c_ext.src_id = sm.src_id
INNER JOIN `{OUTPUT_NAMESPACE}.charlson_concepts` AS cc
    ON condition.condition_concept_id = cc.concept_id
INNER JOIN `{OUTPUT_NAMESPACE}.charlson_scoring` AS cs
    ON cc.diag_category_id = cs.diag_category_id
WHERE c_ext.src_id LIKE '%EHR site%'
GROUP BY 1, 2, 3
)
SELECT cci.src_hpo_id, 
cci.diag_category_id, 
cci.diag_category_name, 
cnt.total_pt_cnt as dx_pt_cnt,
cci.pt_cnt/cnt.total_pt_cnt AS prevalence
 FROM cci
JOIN (SELECT sm.hpo_id AS src_hpo_id, 
COUNT(DISTINCT co.person_id) AS total_pt_cnt 
FROM `{OMOP_NAMESPACE}.condition_occurrence` co
JOIN `{OMOP_NAMESPACE}.condition_occurrence_ext` mp
USING (condition_occurrence_id)
INNER JOIN `{MAPPING_NAMESPACE}.site_maskings` sm
ON mp.src_id = sm.src_id
WHERE mp.src_id LIKE '%EHR site%'
GROUP BY 1) cnt
ON cci.src_hpo_id = cnt.src_hpo_id
ORDER BY diag_category_id, diag_category_name, src_hpo_id
"""

In [ ]:
cci_df = client.query(query).to_dataframe()

In [ ]:
px.box(data_frame=cci_df, x="diag_category_name", y="prevalence", points="all", hover_data=cci_df.columns, range_y=[0, 0.8])

In [ ]:
px.box(data_frame=cci_df, x="src_hpo_id", y="prevalence", points="all", hover_data=cci_df.columns)

In [ ]:
query_omop_2022 = f"""WITH cci AS
(
SELECT DISTINCT mp.src_hpo_id,
                cs.diag_category_id,
                cs.diag_category_name,
                count(distinct cohort.person_id) as pt_cnt
FROM `{UNIONED_EHR_NAMESPACE}.unioned_ehr_person` AS cohort
INNER JOIN `{UNIONED_EHR_NAMESPACE}.unioned_ehr_condition_occurrence` AS condition
    ON cohort.person_id = condition.person_id
INNER JOIN `{UNIONED_EHR_NAMESPACE}._mapping_person` mp
    ON cohort.person_id = mp.person_id
INNER JOIN `{OUTPUT_NAMESPACE}.charlson_scores_omop_2022` AS cc
    ON condition.condition_concept_id = cc.concept_id
INNER JOIN `{OUTPUT_NAMESPACE}.charlson_scoring` AS cs
    ON cc.diag_category_id = cs.diag_category_id
GROUP BY 1, 2, 3
)
SELECT cci.src_hpo_id, 
cci.diag_category_id, 
cci.diag_category_name, 
cnt.total_pt_cnt as dx_pt_cnt,
cci.pt_cnt/cnt.total_pt_cnt AS prevalence
 FROM cci
JOIN (SELECT mp.src_hpo_id, 
COUNT(DISTINCT co.person_id) AS total_pt_cnt 
FROM `{UNIONED_EHR_NAMESPACE}.unioned_ehr_condition_occurrence` co
JOIN `{UNIONED_EHR_NAMESPACE}._mapping_condition_occurrence` mp
USING (condition_occurrence_id)
GROUP BY 1) cnt
ON CCI.src_hpo_id = cnt.src_hpo_id
ORDER BY diag_category_id, diag_category_name, src_hpo_id
"""

In [ ]:
cci_omop_df_2022 = client.query(query_omop_2022).to_dataframe()

In [ ]:
px.box(data_frame=cci_omop_df_2022, x="diag_category_name", y="prevalence", points="all", hover_data=cci_omop_df_2022.columns, range_y=[0,0.8])

### Correlation between CCI and Loyalty Score

In [ ]:
cci_index_query = f"""
SELECT ls.patient_num_pid, ls.score as loyalty_score, ls.cohort_name, cci.person_id, cci.score as charlson_comorbidity_index, cci.lookback_year as lookback_years
 FROM `{OUTPUT_NAMESPACE}.{COHORT_TABLE}` AS ls
 JOIN `{OUTPUT_NAMESPACE}.charlson_comorbidity_score_per_person` cci
 ON ls.patient_num_pid = cci.person_id
 AND SUBSTR(ls.cohort_name, -6, 6) = cci.lookback_year
"""

In [ ]:
ls_cci_df = client.query(cci_index_query).to_dataframe()

In [ ]:
import matplotlib.pyplot as plt

In [ ]:

def cci_loyalty_score_analysis(cci_df, min_lookback_year=1, max_lookback_year=7):
    cci_loyalty_score_correlation = []
    for lookback_years in range(min_lookback_year, max_lookback_year + 1):
        df = cci_df[cci_df['lookback_years'] == str(lookback_years) + 'years']
        df['loyalty_decile'] = pd.qcut(df['loyalty_score'], 10, labels=False)

        # Plot the distribution of comorbidity index for each decile
        #plt.figure(figsize=(20, 8))
        fig = px.box(df, x='loyalty_decile', y='charlson_comorbidity_index', 
                    title=f'Comorbidity Index by Loyalty Score Decile with {lookback_years} Lookback Years', 
                    labels={'loyalty_decile': 'Loyalty Score Decile', 'charlson_comorbidity_index': 'Charlson Comorbidity Index'})
        fig.write_image(f'loyalty_score_cci/loyalty_score_charlson_comorbidity_lookback_years_{lookback_years}.png', format='png')
        cci_loyalty_score_correlation.append({'lookback_year': lookback_years,
                                              'correlation': df['loyalty_score'].corr(df['charlson_comorbidity_index'])})
    return cci_loyalty_score_correlation

In [ ]:
cci_ls_corr = cci_loyalty_score_analysis(ls_cci_df, 1, 7)

In [ ]:
cci_ls_corr

#### Loyalty Score by Size with 1-7 Lookback Year

In [ ]:
loyalty_score_query = f"""
SELECT sm.hpo_id as src_hpo_id, ms.site_id, ls.patient_num_rid, ls.patient_num_pid, ls.src_id, ls.race, ls.ethnicity, ls.gender, ls.sex_at_birth, ls.self_reported_race_ethnicity, ABS(ls.age) AS age, ls.index_dt, ls.last_visit, ls.score, ls.cohort_name, ds1.answer AS income, ds2.answer AS education
FROM `{OUTPUT_NAMESPACE}.{COHORT_TABLE}` AS ls
LEFT JOIN `{OMOP_NAMESPACE}.person_ext` AS pe
ON ls.patient_num_rid = pe.person_id
LEFT JOIN `{ID_CROSSWALK_TABLE}` AS rid_pid
ON ls.patient_num_rid = rid_pid.research_id
LEFT JOIN `{MAPPING_NAMESPACE}.site_maskings` sm
ON ls.src_id = sm.src_id
LEFT JOIN `{OUTPUT_NAMESPACE}.mask_site_id` as ms
ON sm.hpo_id = ms.src_hpo_id
LEFT JOIN `{SURVEY_NAMESPACE}.ds_survey` AS ds1
ON ds1.person_id = ls.patient_num_pid AND ds1.question_concept_id = 1585375
LEFT JOIN `{SURVEY_NAMESPACE}.ds_survey` AS ds2
ON ds2.person_id = ls.patient_num_pid AND ds2.question_concept_id = 1585940
WHERE ls.EHR_EXIST = 1
--1585375:income
--1585940:education
"""

In [ ]:
ls_df = client.query(loyalty_score_query).to_dataframe()

In [ ]:
ls_df['lookback_years'] = ls_df['cohort_name'].str[-6:]

In [ ]:
# Set figure size
plt.figure(figsize=(10, 6))

# Create boxplot
ax = sns.boxplot(data=ls_df, x="lookback_years", y="score")

ax.set(xlabel='Lookback Years',
    ylabel='Loyalty Score',
    title='Loyalty Score by Lookback Years')

# Set y-axis limits
plt.ylim(-0.3, 1.2)

# Adjust layout for spacing
plt.tight_layout()

# Save the figure
plt.savefig("loyalty_score_by_lookback_years.png", format="png", dpi=300)

plt.show()

In [ ]:
ls_df.groupby('lookback_years')['score'].describe()[['mean', '50%', 'std']].rename(columns={'50%': 'median'}).round(2)

In [ ]:
ls_df_lookback_years_5 = ls_df[ls_df['cohort_name'] == f'{COHORT_LABEL_PREFIX}_{ANALYSIS_END_DATE}_5years']

In [ ]:
ls_df_lookback_years_5.count()

In [ ]:
ls_df_lookback_years_5.groupby(['src_hpo_id', 'src_id']).count().sort_values(by='src_id')['site_id']

In [ ]:
ls_site_size = ls_df_lookback_years_5.groupby('site_id')['patient_num_rid'].count().reset_index()

In [ ]:
ls_df_cnt = ls_df_lookback_years_5.merge(ls_site_size, on='site_id')
ls_df_cnt.rename(columns={'patient_num_rid_y': 'site_count'}, inplace=True)

In [ ]:
ls_df_cnt.columns

In [ ]:
ls_site_count_less_than_1000 = ls_df_cnt[ls_df_cnt['site_count'] < 1000]
ls_site_count_1000_5000 = ls_df_cnt[(ls_df_cnt['site_count'] >= 1000) & (ls_df_cnt['site_count'] < 5000)]
ls_site_count_5000_10000 = ls_df_cnt[(ls_df_cnt['site_count'] >= 5000) & (ls_df_cnt['site_count'] < 10000)]
ls_site_count_more_than_10000 = ls_df_cnt[ls_df_cnt['site_count'] >= 10000]

In [ ]:
def draw_violin_plot(df, name):
    ls_site_size = df[['site_id', 'site_count']].drop_duplicates()
    sorted_sites = [site for site, _ in sorted(zip(ls_site_size['site_id'], ls_site_size['site_count']), key=lambda x: x[1])]
    fig = px.violin(data_frame=df, x="src_hpo_id", y="score", category_orders={"site_id": sorted_sites},
              range_y=[-0.3, 1.2], labels={'score': 'Loyalty Score', 'site_id': 'HPO Site'})

    # Extra wide layout
    fig.update_layout(
        width=2500,  # Significantly wider
        height=800,  # Slightly taller
        margin=dict(l=50, r=50, t=80, b=150),  # Extra bottom margin for labels
        xaxis_tickangle=-45,
        xaxis=dict(
            title_font=dict(size=20),
            tickfont=dict(size=16),
        ),
        yaxis=dict(
            title_font=dict(size=20),
            tickfont=dict(size=16),
        ),
        title=f"Site Loyalty Scores for Size {name} (Sites Ordered by Size)",
        plot_bgcolor='white',  # White background for better visibility
    )

    # Adjust violin properties for more space
    fig.update_traces(
        width=0.7,  # Make each violin narrower (default is 1)
        spanmode="hard",
        jitter=0.2,  # Add some jitter to the points
        pointpos=0,  # Center points
        box_visible=True,  # Show box plots inside violins
        meanline_visible=True,  # Show mean lines
    )

    # Use a fixed gap between violins
    fig.update_layout(
        violingap=0.4,  # Increase gap between violins (0-1)
        violingroupgap=0.2,  # Increase gap between any groups
    )

    # Use plotly's more advanced layout options - bargap also affects violin plots
    fig.update_layout(bargap=0.5)  # Increase the gap between categories

    # Improve the x-axis to spread things out

    fig.update_xaxes(
        tickmode="array",
        tickvals=list(range(len(sorted_sites))),
        ticktext=sorted_sites,
        tickangle=-45,
        title_standoff=40,  # Increase distance between axis and title
    )
    fig.write_image(f'loyalty_score_per_site_by_size/violin_per_site_{name.replace(" ", "_")}.png', scale=2) # Scale=2 doubles the resolution

In [ ]:
draw_violin_plot(ls_site_count_less_than_1000, 'Less Than 1000')
draw_violin_plot(ls_site_count_1000_5000, 'Between 1000 and 5000')
draw_violin_plot(ls_site_count_5000_10000, 'Between 5000 and 10000')
draw_violin_plot(ls_site_count_more_than_10000, 'More Than 10000')

##### Loyalty Score Stats per Site for Lookback Year = 5

In [ ]:
ls_df_lookback_years_1 = ls_df[ls_df['cohort_name'] == f'{COHORT_LABEL_PREFIX}_{ANALYSIS_END_DATE}_1years']

In [ ]:
import pandas as pd

def calculate_hpo_statistics(df):
    """
    Calculate statistics per HPO site from a dataframe with columns:
    src_hpo_id, patient_id, hpo_size, score

    Returns a dataframe with statistics per HPO site:
    - min score
    - max score
    - mean score
    - median score
    - patient count
    """
    # Group by HPO site
    hpo_stats = df.groupby('src_hpo_id').agg(
        min_score=('score', 'min'),
        max_score=('score', 'max'),
        mean_score=('score', 'mean'),
        median_score=('score', 'median'),
        patient_count=('patient_num_rid', 'nunique')  # Count unique patients per HPO
    ).reset_index()

    # Round floating point numbers for better readability
    hpo_stats['mean_score'] = hpo_stats['mean_score'].round(2)
    hpo_stats['median_score'] = hpo_stats['median_score'].round(2)

    # Sort by HPO ID for consistency
    hpo_stats = hpo_stats.sort_values('src_hpo_id')

    return hpo_stats

def save_hpo_stats_to_csv(stats_df, output_path='hpo_site_statistics_1year.csv'):
    """
    Save the HPO statistics dataframe to a CSV file
    """
    stats_df.to_csv(output_path, index=False)
    print(f"Statistics saved to {output_path}")

In [ ]:
calculate_hpo_statistics(ls_df_lookback_years_1)

In [ ]:
save_hpo_stats_to_csv(ls_df_lookback_years_5)

##### Loyalty Score by Site by Lookback Years

In [ ]:
ls_df['lookback_years'] = ls_df['cohort_name'].str[-6:]

In [ ]:
ls_df_site = ls_df.groupby(['src_hpo_id', 'lookback_years'])['score'].median().reset_index()

In [ ]:
ls_df_site_1years = ls_df_site[ls_df_site['lookback_years'] == '1years']
print(ls_df_site_1years[ls_df_site_1years['score'] < 0.2].shape[0])
print(ls_df_site_1years[ls_df_site_1years['score'] < 0.2].shape[0] / len(ls_df_site.src_hpo_id.unique()))

In [ ]:
ls_df_site_3years = ls_df_site[ls_df_site['lookback_years'] == '3years']
print(ls_df_site_3years[ls_df_site_3years['score'] < 0.2].shape[0])
print(ls_df_site_3years[ls_df_site_3years['score'] < 0.2].shape[0] / len(ls_df_site.src_hpo_id.unique()))

In [ ]:
ls_df_site_5years = ls_df_site[ls_df_site['lookback_years'] == '5years']
print(ls_df_site_5years[ls_df_site_5years['score'] < 0.2].shape[0])
print(ls_df_site_5years[ls_df_site_5years['score'] < 0.2].shape[0] / len(ls_df_site.src_hpo_id.unique()))

In [ ]:
ls_df_site_7years = ls_df_site[ls_df_site['lookback_years'] == '7years']
print(ls_df_site_7years[ls_df_site_7years['score'] < 0.2].shape[0])
print(ls_df_site_7years[ls_df_site_7years['score'] < 0.2].shape[0] / len(ls_df_site.src_hpo_id.unique()))

##### Loyalty Score by Lookback Years and Demographic subgroups

In [ ]:
import numpy as np

In [ ]:
def cohort_age_group(df):
    age_bins = [17, 35, 50, 65, np.inf]
    age_labels = ['18-34', '35-49', '50-64', '65+']
    df['age_group'] = pd.cut(df['age'], bins=age_bins, labels=age_labels, right=False)
    return df

In [ ]:
ls_df = cohort_age_group(ls_df)

In [ ]:
ls_df.columns

In [ ]:
five_year = ls_df[ls_df['lookback_years'] == '5years']

five_year[five_year['age_group'].isna()][
    ['patient_num_rid', 'age']
].sort_values('age')

In [ ]:
ls_df_no_nulls = ls_df[['patient_num_rid', 'site_id', 'src_id', 'score', 'lookback_years', 'age_group', 
                        'gender', 'race', 'ethnicity', 'sex_at_birth',  'self_reported_race_ethnicity',
                        'income', 'education']].replace(np.NaN, 'Unknown')

In [ ]:
ls_df_5years = ls_df[ls_df['lookback_years'] == '5years'].groupby('patient_num_rid').count().reset_index()

In [ ]:
# No duplicates except participants who got submitted by multiple sites
ls_df_5years[ls_df_5years['score'] > 1]

In [ ]:
ls_df_no_nulls[ls_df_no_nulls['lookback_years'] == '5years'].patient_num_rid.unique()

In [ ]:
ls_df_no_nulls[ls_df_no_nulls['lookback_years'] == '5years']['sex_at_birth'].unique()

In [ ]:
ls_df_no_nulls['self_reported_race_ethnicity'].unique()

In [ ]:
demo_results = pd.DataFrame()
for col in ['age_group', 'sex_at_birth', 'self_reported_race_ethnicity', 'income', 'education']:
    if col == 'sex_at_birth':
        conditions = [
            ls_df_no_nulls['sex_at_birth'].isin(['Male']),
            ls_df_no_nulls['sex_at_birth'].isin(['Female']),
            ls_df_no_nulls['sex_at_birth'].isin(['Intersex']),
            ls_df_no_nulls['sex_at_birth'].isin(['Sex At Birth: Sex At Birth None Of These', 'I prefer not to answer', 'PMI: Skip', 'No matching concept', 'Unknown'])
        ]
        choices = ['Male', 'Female', 'Intersex', 'Unknown']
        ls_df_no_nulls['sex_at_birth'] = np.select(conditions, choices)
    elif col == 'self_reported_race_ethnicity':
        conditions = [
            ls_df_no_nulls['self_reported_race_ethnicity'].isin(['White']),
            ls_df_no_nulls['self_reported_race_ethnicity'].isin(['Black or African American']),
            ls_df_no_nulls['self_reported_race_ethnicity'].isin(['Asian']),
            ls_df_no_nulls['self_reported_race_ethnicity'].isin(['What Race Ethnicity: Hispanic']),
            ls_df_no_nulls['self_reported_race_ethnicity'].isin(['American Indian or Alaska Native', 'Middle Eastern or North African', 'Native Hawaiian or Other Pacific Islander', 'More than one population']),
            ls_df_no_nulls['self_reported_race_ethnicity'].isin(['I prefer not to answer', 'None of these', 'PMI: Skip', 'Unknown', 'No matching concept', 'None Indicated'])
        ]
        choices = ['White', 'Black or African American', 'Asian', 'Hispanic', 'Other', 'Unknown']
        ls_df_no_nulls['self_reported_race_ethnicity'] = np.select(conditions, choices)
    elif col == 'income':
        conditions = [
            ls_df_no_nulls['income'].isin(['Annual Income: less 10k']),
            ls_df_no_nulls['income'].isin(['Annual Income: 10k 25k']),
            ls_df_no_nulls['income'].isin(['Annual Income: 25k 35k']),
            ls_df_no_nulls['income'].isin(['Annual Income: 35k 50k']),
            ls_df_no_nulls['income'].isin(['Annual Income: 50k 75k']),
            ls_df_no_nulls['income'].isin(['Annual Income: 75k 100k']),   
            ls_df_no_nulls['income'].isin(['Annual Income: 100k 150k']),
            ls_df_no_nulls['income'].isin(['Annual Income: 150k 200k']),
            ls_df_no_nulls['income'].isin(['Annual Income: more 200k']),
            ls_df_no_nulls['income'].isin(['PMI: Prefer Not To Answer', 'PMI: Skip', 0, '0', 'No matching concept', 'Unknown'])
        ]
        choices = ['less than 10k', '10k to 25k', '25k to 35k', '35k to 50k', '50k to 75k', '75k to 100k', '100k to 150k', '150k to 200k', 'more than 200k', 'Unknown']
        ls_df_no_nulls['income'] = np.select(conditions, choices)
    elif col == 'education':
        conditions = [
            ls_df_no_nulls['education'].isin(['Highest Grade: Never Attended', 'Highest Grade: One Through Four', 'Highest Grade: Five Through Eight', 'Highest Grade: Nine Through Eleven']),
            ls_df_no_nulls['education'].isin(['Highest Grade: Twelve Or GED']),
            ls_df_no_nulls['education'].isin(['Highest Grade: College One to Three']),
            ls_df_no_nulls['education'].isin(['Highest Grade: College Graduate']),
            ls_df_no_nulls['education'].isin(['Highest Grade: Advanced Degree']),
            ls_df_no_nulls['education'].isin(['PMI: Prefer Not To Answer', 'PMI: Skip', 0, '0', 'Unknown'])
        ]
        choices = ['Less than GED', 'Twelve Or GED', 'College One to Three', 'College Graduate', 'Advanced Degree', 'Unknown']
        ls_df_no_nulls['education'] = np.select(conditions, choices)
    #patient count are the same across all lookback years, so we use 5 years as the reference
    col_count_result = ls_df_no_nulls[ls_df_no_nulls['lookback_years'] == '5years'].groupby([col, 'lookback_years'])['patient_num_rid'].count().reset_index()
    col_count_result = col_count_result.pivot(index=col, columns='lookback_years', values='patient_num_rid').reset_index().rename_axis(None, axis=1)
    col_count_result.rename(columns={col: 'sub-categories', '5years': 'Count'}, inplace=True)
    print(col_count_result)
    col_median_result = ls_df_no_nulls.groupby([col, 'lookback_years'])['score'].median().reset_index()
    col_median_result = col_median_result.pivot(index=col, columns='lookback_years', values='score').reset_index().rename_axis(None, axis=1)
    col_median_result.rename(columns={col: 'sub-categories'}, inplace=True)
    
    col_result = col_count_result.merge(col_median_result, on='sub-categories', how='inner')
    col_result.insert(0, 'variable', col)
    demo_results = pd.concat([demo_results, col_result], axis=0)

In [ ]:
demo_results

In [ ]:
group_sums = demo_results.groupby('variable')['Count'].transform('sum')

In [ ]:
group_sums

In [ ]:
demo_results['Count Percentage'] = (demo_results['Count'] / group_sums) * 100
demo_results['count_percentage'] = demo_results.apply(lambda row: f"{row['Count']} ({row['Count Percentage']:.2f}%)", axis=1)
demo_results.drop(['Count', 'Count Percentage'], axis=1, inplace=True)

In [ ]:
demo_results

In [ ]:
demo_results.to_excel('loyalty_score_by_demo_groups.xlsx', index=False)

#### Linechart for loyalty score by demo groups

In [ ]:
def draw_linegraph(df, group_name):
    group = df[df['variable'] == group_name.lower().replace(' ', '_')]
    group = group.sort_values(by='1years')

    # Prepare x and y values
    years = ['1years', '2years', '3years', '4years', '5years', '6years', '7years']

    # Plot
    plt.figure(figsize=(10, 6))

    for idx, row in group.iterrows():
        plt.plot(years, row[years], marker='o', label=row['sub-categories'])

    plt.title(f'Median Loyalty Score by Lookback Years and {group_name}')
    plt.xlabel('Years')
    plt.ylabel('Median Loyalty Score')
    plt.legend(title=group_name)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f"median_loyalty_score_by_{group_name}.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
draw_linegraph(demo_results, 'Age Group')

In [ ]:
draw_linegraph(demo_results, 'Sex at Birth')

In [ ]:
draw_linegraph(demo_results, 'Income')

In [ ]:
draw_linegraph(demo_results, 'Education')

In [ ]:
demo_results

In [ ]:
draw_linegraph(demo_results, 'Self reported race ethnicity')

### Excluded patients breakdwon

In [ ]:
excluded_patient_query = f"""
WITH
  ehr_flag AS (
    SELECT DISTINCT
      person_id,
      1 as ehr_flag,
      'condition' as omop_domain,
      src_id
      FROM `{OMOP_NAMESPACE}.condition_occurrence`
      JOIN `{OMOP_NAMESPACE}.condition_occurrence_ext`
      USING(condition_occurrence_id)
      WHERE src_id LIKE '%EHR site%'
  UNION ALL
    SELECT DISTINCT
      person_id,
      1 as ehr_flag,
      'drug' as omop_domain,
      src_id
      FROM `{OMOP_NAMESPACE}.drug_exposure`
      JOIN `{OMOP_NAMESPACE}.drug_exposure_ext`
      USING(drug_exposure_id)
      WHERE src_id LIKE '%EHR site%'
  UNION ALL
    SELECT DISTINCT
      person_id,
      1 as ehr_flag,
      'measurement' as omop_domain,
      src_id
      FROM `{OMOP_NAMESPACE}.measurement`
      JOIN `{OMOP_NAMESPACE}.measurement_ext`
      USING(measurement_id)
      WHERE src_id LIKE '%EHR site%'
  UNION ALL
    SELECT DISTINCT
      person_id,
      1 as ehr_flag,
      'procedure' as omop_domain,
      src_id
      FROM `{OMOP_NAMESPACE}.procedure_occurrence`
      JOIN `{OMOP_NAMESPACE}.procedure_occurrence_ext`
      USING(procedure_occurrence_id)
      WHERE src_id LIKE '%EHR site%'
  UNION ALL
    SELECT DISTINCT
      person_id,
      1 as ehr_flag,
      'visit' as omop_domain,
      src_id
      FROM `{OMOP_NAMESPACE}.visit_occurrence`
      JOIN `{OMOP_NAMESPACE}.visit_occurrence_ext`
      USING(visit_occurrence_id)
      WHERE src_id LIKE '%EHR site%'
  UNION ALL
    SELECT DISTINCT
      person_id,
      1 as ehr_flag,
      'observation' as omop_domain,
      src_id
      FROM `{OMOP_NAMESPACE}.observation`
      JOIN `{OMOP_NAMESPACE}.observation_ext`
      USING(observation_id)
      WHERE src_id LIKE '%EHR site%'
      AND observation_concept_id not in (4013886, 4135376, 4271761)
  ),
  ehr_exist AS (
  SELECT
    DISTINCT person_id,
    src_id,
    COUNT(DISTINCT omop_domain) as ehr_domain_count
  FROM
    ehr_flag
  GROUP BY person_id, src_id)
SELECT DISTINCT p.person_id,
c1.concept_name AS gender,
c2.concept_name AS race,
c3.concept_name AS ethnicity,
c4.concept_name AS sex_at_birth,
c5.concept_name AS self_reported_race_ethnicity, 
ROUND(DATE_DIFF(DATE('{ANALYSIS_END_DATE}'), DATE(p.birth_datetime), DAY)/365) AS age, 
ds1.answer AS income, 
ds2.answer AS education
FROM `{OMOP_NAMESPACE}.person` p
LEFT JOIN `{OUTPUT_NAMESPACE}.{COHORT_TABLE}` AS ls
ON p.person_id = ls.patient_num_rid
JOIN `{OMOP_NAMESPACE}.person_ext` pe
  ON p.person_id = pe.person_id
JOIN `{ID_CROSSWALK_TABLE}` rid
  ON p.person_id = rid.research_id
LEFT JOIN `{OMOP_NAMESPACE}.concept` c1
ON p.gender_concept_id = c1.concept_id
LEFT JOIN  `{OMOP_NAMESPACE}.concept` c2
  ON p.race_concept_id = c2.concept_id
LEFT JOIN  `{OMOP_NAMESPACE}.concept` c3
  ON p.ethnicity_concept_id = c3.concept_id
LEFT JOIN  `{OMOP_NAMESPACE}.concept` c4
  ON pe.sex_at_birth_concept_id = c4.concept_id
LEFT JOIN  `{OMOP_NAMESPACE}.concept` c5
  ON pe.self_reported_category_concept_id = c5.concept_id
LEFT JOIN `{SURVEY_NAMESPACE}.ds_survey` AS ds1
ON ds1.person_id = rid.person_id AND ds1.question_concept_id = 1585375
LEFT JOIN `{SURVEY_NAMESPACE}.ds_survey` AS ds2
ON ds2.person_id = rid.person_id AND ds2.question_concept_id = 1585940
--1585375:income
--1585940:education
LEFT JOIN
  ehr_exist
ON
  p.person_id = ehr_exist.person_id
WHERE ls.patient_num_rid IS NULL
AND ehr_exist.ehr_domain_count >= 1
"""

In [ ]:
excluded_patient_df = client.query(excluded_patient_query).to_dataframe()

In [ ]:
excluded_patient_df

In [ ]:
excluded_patient_df.person_id.unique()

In [ ]:
excluded_patient_df = cohort_age_group(excluded_patient_df)

In [ ]:
excluded_patient_df.person_id.unique()

In [ ]:
excluded_patient_df_no_nulls = excluded_patient_df[['person_id', 'age_group', 
                        'gender', 'race', 'ethnicity', 'sex_at_birth',  'self_reported_race_ethnicity',
                        'income', 'education']].replace(np.NaN, 'Unknown')

In [ ]:
demo_results = pd.DataFrame()
for col in ['age_group', 'sex_at_birth', 'race', 'income', 'education']:
    if col == 'sex_at_birth':
        conditions = [
            excluded_patient_df_no_nulls['sex_at_birth'].isin(['Male']),
            excluded_patient_df_no_nulls['sex_at_birth'].isin(['Female']),
            excluded_patient_df_no_nulls['sex_at_birth'].isin(['Intersex']),
            excluded_patient_df_no_nulls['sex_at_birth'].isin(['Sex At Birth: Sex At Birth None Of These', 'I prefer not to answer', 'PMI: Skip', 'No matching concept', 'Unknown'])
        ]
        choices = ['Male', 'Female', 'Intersex', 'Unknown']
        excluded_patient_df_no_nulls['sex_at_birth'] = np.select(conditions, choices)
    elif col == 'race':
        conditions = [
            excluded_patient_df_no_nulls['race'].isin(['White']),
            excluded_patient_df_no_nulls['race'].isin(['Black or African American']),
            excluded_patient_df_no_nulls['race'].isin(['Asian']),
            excluded_patient_df_no_nulls['race'].isin(['American Indian or Alaska Native', 'Middle Eastern or North African', 'Native Hawaiian or Other Pacific Islander', 'More than one population']),
            excluded_patient_df_no_nulls['race'].isin(['I prefer not to answer', 'None of these', 'PMI: Skip', 'Unknown', 'No matching concept', 'None Indicated'])
        ]
        choices = ['White', 'Black or African American', 'Asian', 'Other', 'Unknown']
        excluded_patient_df_no_nulls['race'] = np.select(conditions, choices)
    elif col == 'income':
        conditions = [
            excluded_patient_df_no_nulls['income'].isin(['Annual Income: less 10k']),
            excluded_patient_df_no_nulls['income'].isin(['Annual Income: 10k 25k']),
            excluded_patient_df_no_nulls['income'].isin(['Annual Income: 25k 35k']),
            excluded_patient_df_no_nulls['income'].isin(['Annual Income: 35k 50k']),
            excluded_patient_df_no_nulls['income'].isin(['Annual Income: 50k 75k']),
            excluded_patient_df_no_nulls['income'].isin(['Annual Income: 75k 100k']),   
            excluded_patient_df_no_nulls['income'].isin(['Annual Income: 100k 150k']),
            excluded_patient_df_no_nulls['income'].isin(['Annual Income: 150k 200k']),
            excluded_patient_df_no_nulls['income'].isin(['Annual Income: more 200k']),
            excluded_patient_df_no_nulls['income'].isin(['PMI: Prefer Not To Answer', 'PMI: Skip', 0, '0', 'No matching concept', 'Unknown'])
        ]
        choices = ['less than 10k', '10k to 25k', '25k to 35k', '35k to 50k', '50k to 75k', '75k to 100k', '100k to 150k', '150k to 200k', 'more than 200k', 'Unknown']
        excluded_patient_df_no_nulls['income'] = np.select(conditions, choices)
    elif col == 'education':
        conditions = [
            excluded_patient_df_no_nulls['education'].isin(['Highest Grade: Never Attended', 'Highest Grade: One Through Four', 'Highest Grade: Five Through Eight', 'Highest Grade: Nine Through Eleven']),
            excluded_patient_df_no_nulls['education'].isin(['Highest Grade: Twelve Or GED']),
            excluded_patient_df_no_nulls['education'].isin(['Highest Grade: College One to Three']),
            excluded_patient_df_no_nulls['education'].isin(['Highest Grade: College Graduate']),
            excluded_patient_df_no_nulls['education'].isin(['Highest Grade: Advanced Degree']),
            excluded_patient_df_no_nulls['education'].isin(['PMI: Prefer Not To Answer', 'PMI: Skip', 0, '0', 'Unknown'])
        ]
        choices = ['Less than GED', 'Twelve Or GED', 'College One to Three', 'College Graduate', 'Advanced Degree', 'Unknown']
        excluded_patient_df_no_nulls['education'] = np.select(conditions, choices)

In [ ]:
age_group = excluded_patient_df_no_nulls.groupby('age_group').count()['person_id'].reset_index()

In [ ]:
age_group

In [ ]:
group_sums = 28352

In [ ]:
age_group['Count Percentage'] = (age_group['person_id'] / group_sums) * 100
age_group['count_percentage'] = age_group.apply(lambda row: f"{row['person_id']} ({row['Count Percentage']:.2f}%)", axis=1)
age_group.drop(['person_id', 'Count Percentage'], axis=1, inplace=True)

In [ ]:
age_group

In [ ]:
sex_at_birth = excluded_patient_df_no_nulls.groupby('sex_at_birth').count()['person_id'].reset_index()

In [ ]:
sex_at_birth['Count Percentage'] = (sex_at_birth['person_id'] / group_sums) * 100
sex_at_birth['count_percentage'] = sex_at_birth.apply(lambda row: f"{row['person_id']} ({row['Count Percentage']:.2f}%)", axis=1)
sex_at_birth.drop(['person_id', 'Count Percentage'], axis=1, inplace=True)

In [ ]:
sex_at_birth

In [ ]:
race = excluded_patient_df_no_nulls.groupby('race').count()['person_id'].reset_index()

In [ ]:
race['Count Percentage'] = (race['person_id'] / group_sums) * 100
race['count_percentage'] = race.apply(lambda row: f"{row['person_id']} ({row['Count Percentage']:.2f}%)", axis=1)
race.drop(['person_id', 'Count Percentage'], axis=1, inplace=True)
race

In [ ]:
excluded_patient_df_no_nulls.columns

In [ ]:
income = excluded_patient_df_no_nulls.groupby('income').count()['person_id'].reset_index()

In [ ]:
income['Count Percentage'] = (income['person_id'] / group_sums) * 100
income['count_percentage'] = income.apply(lambda row: f"{row['person_id']} ({row['Count Percentage']:.2f}%)", axis=1)
income.drop(['person_id', 'Count Percentage'], axis=1, inplace=True)
income

In [ ]:
education = excluded_patient_df_no_nulls.groupby('education').count()['person_id'].reset_index()

In [ ]:
education['Count Percentage'] = (education['person_id'] / group_sums) * 100
education['count_percentage'] = education.apply(lambda row: f"{row['person_id']} ({row['Count Percentage']:.2f}%)", axis=1)
education.drop(['person_id', 'Count Percentage'], axis=1, inplace=True)
education